In [ ]:
# %% [markdown]
# # Sri Lanka Fundamental Rights Document Processor
#
# This notebook processes the FUNDAMENTAL RIGHTS.pdf file and creates the data files needed for your legal summarizer project.

# %% [code]
# Install required packages
!pip install pypdf2 PyMuPDF pdfplumber nltk

# %% [code]
import json
import re
import os
from typing import Dict, List, Any, Tuple
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter

# Download NLTK data
nltk.download('punkt_tab')
nltk.download('punkt')

# %% [markdown]
# ## Step 1: Upload your FUNDAMENTAL RIGHTS.pdf file

# %% [code]
from google.colab import files

print("Please upload your FUNDAMENTAL RIGHTS.pdf file")
uploaded = files.upload()

# Get the uploaded file name
pdf_file = None
for filename in uploaded.keys():
    if filename.lower().endswith('.pdf'):
        pdf_file = filename
        print(f"✓ Uploaded: {pdf_file}")
        break

if not pdf_file:
    print("❌ No PDF file found in upload")
    # List files in current directory
    print("\nFiles in current directory:")
    print(os.listdir('.'))

# %% [markdown]
# ## Step 2: Extract and clean text from PDF

# %% [code]
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from PDF using multiple methods."""
    text = ""

    # Try PyMuPDF (fitz) first - usually best for text extraction
    try:
        import fitz
        doc = fitz.open(pdf_path)
        for page_num, page in enumerate(doc):
            page_text = page.get_text()
            if page_text:
                # Add page marker for reference
                text += f"\n===== Page {page_num + 1} =====\n{page_text}"
        doc.close()
        print(f"✓ Extracted {len(doc)} pages using PyMuPDF")
        return text
    except Exception as e:
        print(f"PyMuPDF failed: {e}")

    # Try pdfplumber as fallback
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path, strict=False) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += f"\n===== Page {page_num + 1} =====\n{page_text}"
        print(f"✓ Extracted text using pdfplumber")
        return text
    except Exception as e:
        print(f"pdfplumber failed: {e}")

    # Try PyPDF2 as last resort
    try:
        import PyPDF2
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                page_text = page.extract_text()
                if page_text:
                    text += f"\n===== Page {page_num + 1} =====\n{page_text}"
        print(f"✓ Extracted text using PyPDF2")
        return text
    except Exception as e:
        print(f"PyPDF2 failed: {e}")

    return text

# Extract text if PDF was uploaded
if pdf_file:
    raw_text = extract_text_from_pdf(pdf_file)
    print(f"\n✓ Extracted {len(raw_text)} characters")

    # Save raw text
    with open('fundamental_rights_raw.txt', 'w', encoding='utf-8') as f:
        f.write(raw_text)

    # Show preview
    print("\nPreview of extracted text (first 1000 characters):")
    print("-" * 80)
    print(raw_text[:1000])
    print("-" * 80)
else:
    print("No PDF to process")

# %% [markdown]
# ## Step 3: Parse and structure the fundamental rights document

# %% [code]
def parse_fundamental_rights_document(text: str) -> Dict[str, Any]:
    """Parse the fundamental rights document into structured data."""

    # First, remove page markers and combine
    pages = re.split(r'===== Page \d+ =====', text)
    pages = [page.strip() for page in pages if page.strip()]

    # Combine all pages
    full_text = " ".join(pages)

    # Clean up text
    full_text = re.sub(r'\s+', ' ', full_text)  # Normalize whitespace
    full_text = re.sub(r'-\s*\n\s*', '', full_text)  # Fix hyphenated words

    print(f"Full text length: {len(full_text)} characters")
    print(f"Word count: {len(full_text.split())} words")

    # Split into sentences
    sentences = sent_tokenize(full_text)
    print(f"Total sentences: {len(sentences)}")

    # Find all articles - they start with numbers
    articles = {}
    current_article = None
    current_text = []
    article_pattern = re.compile(r'^(\d+[A-Z]?\.?)\s+(.*)')

    for sentence in sentences:
        # Check if this is an article number
        match = article_pattern.match(sentence.strip())

        if match:
            # Save previous article if exists
            if current_article is not None and current_text:
                articles[current_article] = {
                    "text": " ".join(current_text),
                    "sentences": current_text.copy(),
                    "title": extract_article_title(current_text[0] if current_text else "")
                }

            # Start new article
            current_article = match.group(1).rstrip('.')
            current_text = [sentence.strip()]
        elif current_article is not None:
            # Continue current article
            current_text.append(sentence.strip())

    # Save the last article
    if current_article is not None and current_text:
        articles[current_article] = {
            "text": " ".join(current_text),
            "sentences": current_text.copy(),
            "title": extract_article_title(current_text[0] if current_text else "")
        }

    # If no articles found with pattern, try to find them by content
    if not articles:
        print("No articles found with pattern matching. Trying content-based parsing...")
        articles = parse_by_content(sentences)

    return articles, sentences, full_text

def extract_article_title(first_sentence: str) -> str:
    """Extract a meaningful title from the first sentence of an article."""
    # Remove article number
    title = re.sub(r'^\d+[A-Z]?\.?\s*', '', first_sentence)

    # Take first 15 words max
    words = title.split()[:15]
    title_text = " ".join(words)

    # Add ellipsis if truncated
    if len(title.split()) > 15:
        title_text += "..."

    return title_text

def parse_by_content(sentences: List[str]) -> Dict[str, Any]:
    """Parse articles based on content when pattern matching fails."""
    articles = {}
    current_article = None
    current_text = []
    article_counter = 1

    for sentence in sentences:
        # Look for sentences that might start articles
        # Usually they contain "Every person" or "No person" or mention rights
        if re.search(r'(Every person|No person|All persons|Every citizen|No citizen)', sentence, re.IGNORECASE):
            # Save previous article
            if current_article is not None and current_text:
                articles[current_article] = {
                    "text": " ".join(current_text),
                    "sentences": current_text.copy(),
                    "title": extract_article_title(current_text[0] if current_text else "")
                }

            # Start new article
            current_article = str(article_counter)
            current_text = [sentence]
            article_counter += 1
        elif current_article is not None:
            # Continue current article
            current_text.append(sentence)
        elif re.search(r'(freedom|right|entitled|protected|guaranteed)', sentence, re.IGNORECASE):
            # This might be the start of the document
            current_article = "1"
            current_text = [sentence]

    # Save the last article
    if current_article is not None and current_text:
        articles[current_article] = {
            "text": " ".join(current_text),
            "sentences": current_text.copy(),
            "title": extract_article_title(current_text[0] if current_text else "")
        }

    return articles

# Parse the document
if 'raw_text' in locals():
    articles, all_sentences, full_text = parse_fundamental_rights_document(raw_text)

    print(f"\n✓ Parsed {len(articles)} articles")

    # Show sample articles
    print("\nSample articles:")
    for i, (art_num, art_data) in enumerate(list(articles.items())[:5]):
        print(f"\nArticle {art_num}:")
        print(f"  Title: {art_data.get('title', 'N/A')}")
        print(f"  Sentences: {len(art_data.get('sentences', []))}")
        print(f"  Preview: {art_data['text'][:150]}...")

# %% [markdown]
# ## Step 4: Create processed_fundamental_rights.json (for fundamental rights)

# %% [code]
def create_processed_fundamental_rights(articles: Dict[str, Any]) -> Dict[str, Any]:
    """Create processed_fundamental_rights.json structure for fundamental rights."""

    processed_data = {}
    passage_id_counter = 1

    for art_num, art_data in articles.items():
        article_key = f"fundamental_rights_article_{art_num}"

        # Create passages for each sentence in the article
        passages = []

        for idx, sentence in enumerate(art_data.get('sentences', [])):
            passage = {
                "id": str(passage_id_counter),
                "article": art_num,
                "text": sentence,
                "source": "FUNDAMENTAL RIGHTS.pdf",
                "document_type": "Fundamental Rights",
                "language": "en",
                "metadata": {
                    "chapter": "FUNDAMENTAL RIGHTS",
                    "article_number": art_num,
                    "sentence_index": idx,
                    "word_count": len(sentence.split()),
                    "is_fundamental_right": True
                }
            }
            passages.append(passage)
            passage_id_counter += 1

        # Add the article as a whole document
        processed_data[article_key] = {
            "article_number": art_num,
            "title": art_data.get('title', f'Article {art_num} - Fundamental Rights'),
            "full_text": art_data.get('text', ''),
            "sentence_count": len(art_data.get('sentences', [])),
            "word_count": len(art_data.get('text', '').split()),
            "passages": passages,
            "source_file": "FUNDAMENTAL RIGHTS.pdf",
            "document_type": "Fundamental Rights",
            "language": "en",
            "chapter": "FUNDAMENTAL RIGHTS",
            "is_fundamental_right": True
        }

    return processed_data

if 'articles' in locals():
    processed_data = create_processed_fundamental_rights(articles)

    # Save to JSON
    with open('processed_fundamental_rights.json', 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Created processed_fundamental_rights.json")
    print(f"  Articles: {len(processed_data)}")

    # Calculate total passages
    total_passages = sum(len(data['passages']) for data in processed_data.values())
    print(f"  Total passages: {total_passages}")

    # Show sample
    first_key = list(processed_data.keys())[0]
    print(f"\nSample entry ({first_key}):")
    print(f"  Title: {processed_data[first_key].get('title', 'N/A')}")
    print(f"  Passages: {len(processed_data[first_key].get('passages', []))}")

# %% [markdown]
# ## Step 5: Create fundamental_rights_articles.json   (for fundamental rights)

# %% [code]
def create_fundamental_rights_articles(articles: Dict[str, Any], full_text: str) -> Dict[str, Any]:
    """Create fundamental_rights_articles.json   structure for fundamental rights."""

    fundamental_rights_articles = {}

    # Analyze full text for common themes
    all_text = " ".join([art_data['text'] for art_data in articles.values()])
    common_terms = extract_common_terms(all_text)

    for art_num, art_data in articles.items():
        article_text = art_data.get('text', '')

        # Extract key information
        category = categorize_fundamental_right(article_text)
        rights = extract_specific_rights(article_text)
        restrictions = extract_restrictions(article_text)

        # Create comprehensive article entry
        fundamental_rights_articles[art_num] = {
            "article_number": art_num,
            "title": art_data.get('title', f'Article {art_num}'),
            "description": article_text[:200] + "..." if len(article_text) > 200 else article_text,
            "full_text": article_text,
            "category": category,
            "rights_protected": rights,
            "restrictions": restrictions,
            "applicability": determine_applicability(article_text),
            "legal_terms": extract_legal_terms(article_text, common_terms),
            "keywords": extract_keywords(article_text),
            "source": "FUNDAMENTAL RIGHTS.pdf",
            "document_type": "Fundamental Rights",
            "language": "en",
            "word_count": len(article_text.split()),
            "sentence_count": len(art_data.get('sentences', [])),
            "is_fundamental_right": True
        }

    return fundamental_rights_articles

def extract_common_terms(text: str, top_n: int = 20) -> List[str]:
    """Extract most common terms from the text."""
    words = word_tokenize(text.lower())
    # Remove stopwords and punctuation
    stopwords = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'being'])
    filtered_words = [word for word in words if word.isalpha() and word not in stopwords]

    # Get most common words
    word_counts = Counter(filtered_words)
    return [word for word, count in word_counts.most_common(top_n)]

def categorize_fundamental_right(text: str) -> str:
    """Categorize the fundamental right based on content."""
    text_lower = text.lower()

    if any(term in text_lower for term in ['thought', 'conscience', 'religion', 'belief']):
        return "Freedom of Thought, Conscience and Religion"
    elif any(term in text_lower for term in ['torture', 'cruel', 'inhuman', 'degrading']):
        return "Freedom from Torture and Inhuman Treatment"
    elif any(term in text_lower for term in ['equal', 'equality', 'discrimination', 'discriminated']):
        return "Equality and Non-discrimination"
    elif any(term in text_lower for term in ['arrest', 'detention', 'punishment', 'trial', 'innocent']):
        return "Due Process and Fair Trial Rights"
    elif any(term in text_lower for term in ['speech', 'expression', 'assembly', 'association', 'movement']):
        return "Freedom of Expression and Association"
    elif any(term in text_lower for term in ['information', 'access to information']):
        return "Right to Information"
    else:
        return "General Fundamental Right"

def extract_specific_rights(text: str) -> List[str]:
    """Extract specific rights mentioned in the article."""
    rights = []

    # Patterns to identify rights
    patterns = [
        r'right to ([a-zA-Z\s]+(?:and|or)[a-zA-Z\s]+)',
        r'right of ([a-zA-Z\s]+)',
        r'freedom of ([a-zA-Z\s]+)',
        r'freedom to ([a-zA-Z\s]+)',
        r'entitled to ([a-zA-Z\s]+)'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            if isinstance(match, str):
                rights.append(match.strip())

    return list(set(rights))[:10]  # Return unique rights

def extract_restrictions(text: str) -> List[str]:
    """Extract restrictions mentioned in the article."""
    restrictions = []
    sentences = sent_tokenize(text)

    for sentence in sentences:
        sentence_lower = sentence.lower()
        if any(term in sentence_lower for term in [
            'subject to', 'prescribed by law', 'restriction', 'restrictions',
            'necessary', 'democratic society', 'national security',
            'public order', 'public health', 'morality'
        ]):
            restrictions.append(sentence.strip())

    return restrictions[:5]

def determine_applicability(text: str) -> str:
    """Determine who the right applies to."""
    text_lower = text.lower()

    if 'every person' in text_lower:
        return "All persons"
    elif 'every citizen' in text_lower:
        return "Citizens only"
    elif 'no person' in text_lower:
        return "All persons (negative obligation)"
    elif 'all persons' in text_lower:
        return "All persons"
    else:
        return "To be determined"

def extract_legal_terms(text: str, common_terms: List[str]) -> List[str]:
    """Extract legal terms from the text."""
    legal_terms = []
    text_lower = text.lower()

    # Check for common legal terms
    legal_keywords = [
        'due process', 'equal protection', 'fair trial', 'presumed innocent',
        'arbitrary arrest', 'degrading treatment', 'freedom of conscience',
        'procedure established by law', 'retrospective legislation',
        'subordinate legislation', 'executive action', 'constitutional remedy',
        'fundamental right', 'human dignity', 'judicial review'
    ]

    for term in legal_keywords:
        if term in text_lower:
            legal_terms.append(term)

    # Also include top common terms that are substantive
    for term in common_terms[:10]:
        if len(term) > 4:  # Avoid very short words
            legal_terms.append(term)

    return list(set(legal_terms))[:15]

def extract_keywords(text: str) -> List[str]:
    """Extract keywords from the text."""
    keywords = []
    sentences = sent_tokenize(text)

    # Take important phrases from first few sentences
    for i, sentence in enumerate(sentences[:3]):
        words = word_tokenize(sentence.lower())
        # Get nouns and adjectives (simple heuristic)
        important_words = [word for word in words if len(word) > 4][:5]
        keywords.extend(important_words)

    return list(set(keywords))[:10]

if 'articles' in locals() and 'full_text' in locals():
    fundamental_rights_articles = create_fundamental_rights_articles(articles, full_text)

    # Save to JSON
    with open('fundamental_rights_articles.json  ', 'w', encoding='utf-8') as f:
        json.dump(fundamental_rights_articles, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Created fundamental_rights_articles.json  ")
    print(f"  Total articles: {len(fundamental_rights_articles)}")

    # Show sample
    print("\nSample article (Article 1):")
    if '1' in fundamental_rights_articles:
        sample = fundamental_rights_articles['1']
        print(f"  Title: {sample.get('title', 'N/A')}")
        print(f"  Category: {sample.get('category', 'N/A')}")
        print(f"  Rights: {', '.join(sample.get('rights_protected', [])[:3])}")

# %% [markdown]
# ## Step 6: Create fundamental_rights_patterns.json

# %% [code]
def create_enriched_patterns(articles: Dict[str, Any], fundamental_rights_articles: Dict[str, Any]) -> Dict[str, List[str]]:
    """Create patterns for fundamental rights detection."""

    patterns = {}

    for art_num, art_data in articles.items():
        article_patterns = []
        text = art_data.get('text', '')
        title = art_data.get('title', '')

        # Get additional data from fundamental_rights_articles if available
        if art_num in fundamental_rights_articles:
            const_data = fundamental_rights_articles[art_num]
            keywords = const_data.get('keywords', [])
            rights = const_data.get('rights_protected', [])
            category = const_data.get('category', '')
        else:
            keywords = []
            rights = []
            category = ''

        # 1. Add title as pattern
        if title:
            article_patterns.append(title.lower())

        # 2. Add category as pattern
        if category:
            article_patterns.append(category.lower())

        # 3. Add keywords
        for keyword in keywords:
            if isinstance(keyword, str) and 2 <= len(keyword.split()) <= 4:
                article_patterns.append(keyword.lower())

        # 4. Add rights
        for right in rights:
            if isinstance(right, str):
                article_patterns.append(right.lower())

        # 5. Extract key phrases from text (first few sentences)
        sentences = art_data.get('sentences', [])
        for sentence in sentences[:2]:  # First 2 sentences
            # Clean and add meaningful phrases
            clean_sent = re.sub(r'[\W_]+', ' ', sentence).strip() # Using [\W_]+ to remove punctuation and underscores
            words = clean_sent.split()
            if 4 <= len(words) <= 10:
                article_patterns.append(clean_sent.lower())

        # 6. Generate specific patterns based on article number
        article_patterns.extend(generate_article_specific_patterns(art_num, text))

        # 7. Add common variations
        article_patterns.extend(generate_common_variations(text))

        # Remove duplicates, empty strings, and limit length
        unique_patterns = list(set([
            p.strip() for p in article_patterns
            if p.strip() and len(p.strip()) > 5
        ]))

        # Sort by length (shorter patterns first)
        unique_patterns.sort(key=len)

        if unique_patterns:
            patterns[art_num] = unique_patterns[:25]  # Limit to 25 patterns per article

    return patterns

def generate_article_specific_patterns(art_num: str, text: str) -> List[str]:
    """Generate patterns specific to each article."""
    specific_patterns = []
    text_lower = text.lower()

    # Map article numbers to common patterns based on your PDF content
    if art_num == '1':
        specific_patterns.extend([
            "freedom of thought",
            "freedom of conscience",
            "freedom of religion",
            "religious belief",
            "freedom to adopt religion",
            "right to belief"
        ])
    elif art_num == '2':
        specific_patterns.extend([
            "torture",
            "cruel treatment",
            "inhuman treatment",
            "degrading treatment",
            "cruel punishment",
            "inhuman punishment"
        ])
    elif art_num == '3':
        specific_patterns.extend([
            "equal before the law",
            "equal protection of the law",
            "no discrimination",
            "racial discrimination",
            "religious discrimination",
            "gender discrimination",
            "equality rights"
        ])
    elif art_num == '4':
        specific_patterns.extend([
            "arbitrary arrest",
            "procedure established by law",
            "right to fair trial",
            "presumed innocent",
            "retrospective legislation",
            "due process rights"
        ])
    elif art_num == '5':
        specific_patterns.extend([
            "freedom of speech",
            "freedom of expression",
            "freedom of assembly",
            "freedom of association",
            "freedom of movement",
            "right to residence",
            "cultural rights",
            "language rights"
        ])
    elif art_num == '14A':
        specific_patterns.extend([
            "right to information",
            "access to information",
            "information held by state",
            "transparency right",
            "freedom of information"
        ])
    elif art_num in ['6', '7', '8']:
        specific_patterns.extend([
            "restrictions on rights",
            "limitations on fundamental rights",
            "existing law continues",
            "remedy for rights violation",
            "constitutional remedy"
        ])

    return specific_patterns

def generate_common_variations(text: str) -> List[str]:
    """Generate common variations of phrases in the text."""
    variations = []

    # Look for "Every person is entitled to X" pattern
    every_person_match = re.search(r'Every person is entitled to ([^\.]+)', text, re.IGNORECASE)
    if every_person_match:
        right = every_person_match.group(1).strip()
        variations.append(f"right to {right}")
        variations.append(f"entitled to {right}")

    # Look for "No person shall be X" pattern
    no_person_match = re.search(r'No person shall be ([^\.]+)', text, re.IGNORECASE)
    if no_person_match:
        prohibition = no_person_match.group(1).strip()
        variations.append(f"prohibition of {prohibition}")
        variations.append(f"no {prohibition}")

    # Look for "All persons are X" pattern
    all_persons_match = re.search(r'All persons are ([^\.]+)', text, re.IGNORECASE)
    if all_persons_match:
        status = all_persons_match.group(1).strip()
        variations.append(f"all persons {status}")

    return variations

if 'articles' in locals() and 'fundamental_rights_articles' in locals():
    enriched_patterns = create_enriched_patterns(articles, fundamental_rights_articles)

    # Save to JSON
    with open('fundamental_rights_patterns.json', 'w', encoding='utf-8') as f:
        json.dump(enriched_patterns, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Created fundamental_rights_patterns.json")
    print(f"  Articles with patterns: {len(enriched_patterns)}")

    # Calculate total patterns
    total_patterns = sum(len(patterns) for patterns in enriched_patterns.values())
    print(f"  Total patterns: {total_patterns}")

    # Show sample patterns
    print("\nSample patterns for Article 1:")
    if '1' in enriched_patterns:
        for pattern in enriched_patterns['1'][:8]:
            print(f"  - {pattern}")

# %% [markdown]
# ## Step 7: Create summary statistics and validation

# %% [code]
def create_summary_statistics(articles: Dict[str, Any],
                               processed_data: Dict[str, Any],
                               fundamental_rights_articles: Dict[str, Any],
                               enriched_patterns: Dict[str, Any]) -> Dict[str, Any]:
    """Create summary statistics of the processed data."""

    stats = {
        "document_info": {
            "source_file": "FUNDAMENTAL RIGHTS.pdf",
            "document_type": "Fundamental Rights",
            "processing_date": "2024",
            "language": "en"
        },
        "articles_summary": {
            "total_articles": len(articles),
            "article_numbers": list(articles.keys()),
            "total_sentences": sum(len(art_data.get('sentences', [])) for art_data in articles.values()),
            "total_words": sum(len(art_data.get('text', '').split()) for art_data in articles.values())
        },
        "processed_data_summary": {
            "total_entries": len(processed_data),
            "total_passages": sum(len(data.get('passages', [])) for data in processed_data.values())
        },
        "patterns_summary": {
            "articles_with_patterns": len(enriched_patterns),
            "total_patterns": sum(len(patterns) for patterns in enriched_patterns.values()),
            "average_patterns_per_article": sum(len(patterns) for patterns in enriched_patterns.values()) / max(1, len(enriched_patterns))
        },
        "content_categories": {
            "categories_found": list(set(art.get('category', '') for art in fundamental_rights_articles.values() if art.get('category')))
        }
    }

    return stats

if all(var in locals() for var in ['articles', 'processed_data', 'fundamental_rights_articles', 'enriched_patterns']):
    stats = create_summary_statistics(articles, processed_data, fundamental_rights_articles, enriched_patterns)

    # Save statistics
    with open('processing_statistics.json', 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Created processing_statistics.json")
    print("\n" + "="*60)
    print("PROCESSING SUMMARY")
    print("="*60)

    print(f"\nDocument: {stats['document_info']['source_file']}")
    print(f"Type: {stats['document_info']['document_type']}")

    print(f"\nArticles found: {stats['articles_summary']['total_articles']}")
    print(f"Article numbers: {', '.join(stats['articles_summary']['article_numbers'])}")
    print(f"Total sentences: {stats['articles_summary']['total_sentences']}")
    print(f"Total words: {stats['articles_summary']['total_words']}")

    print(f"\nProcessed data entries: {stats['processed_data_summary']['total_entries']}")
    print(f"Total passages for semantic search: {stats['processed_data_summary']['total_passages']}")

    print(f"\nPatterns for detection: {stats['patterns_summary']['total_patterns']}")
    print(f"Articles with patterns: {stats['patterns_summary']['articles_with_patterns']}")

    print(f"\nCategories identified:")
    for category in stats['content_categories']['categories_found']:
        print(f"  - {category}")

# %% [markdown]
# ## Step 8: Prepare files for download

# %% [code]
# List all generated files
print("\n" + "="*60)
print("GENERATED FILES SUMMARY")
print("="*60)

generated_files = [
    ('processed_fundamental_rights.json', 'Contains structured passages from fundamental rights for semantic search'),
    ('fundamental_rights_articles.json  ', 'Contains detailed fundamental rights article information'),
    ('fundamental_rights_patterns.json', 'Patterns for detecting fundamental rights in text'),
    ('fundamental_rights_raw.txt', 'Raw extracted text from PDF'),
    ('processing_statistics.json', 'Summary statistics of the processed data')
]

for filename, description in generated_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"\n✓ {filename}")
        print(f"  Size: {size:,} bytes")
        print(f"  Description: {description}")
    else:
        print(f"\n⚠️  {filename} - NOT FOUND")

# %% [markdown]
# ## Step 9: Create a zip file for easy download

# %% [code]
# Create a zip file with all generated data
import zipfile

# Create zip file
zip_filename = 'sri_lanka_fundamental_rights_data.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for filename, _ in generated_files:
        if os.path.exists(filename):
            zipf.write(filename)

if os.path.exists(zip_filename):
    size = os.path.getsize(zip_filename)
    print(f"\n✓ Created {zip_filename}")
    print(f"  Zip file size: {size:,} bytes")
else:
    print(f"\n❌ Failed to create zip file")

# %% [markdown]
# ## Step 10: Download instructions

# %% [code]
print("\n" + "="*60)
print("DOWNLOAD INSTRUCTIONS")
print("="*60)

print("\nTo download the generated data:")
print("\nOption 1: Download all files as a zip:")
print("  files.download('sri_lanka_fundamental_rights_data.zip')")

print("\nOption 2: Download individual files:")
for filename, _ in generated_files:
    if os.path.exists(filename):
        print(f"  files.download('{filename}')")

print("\n" + "="*60)
print("HOW TO USE IN YOUR PROJECT")
print("="*60)

print("""
1. Download the generated files using the commands above.

2. Place the files in your project's data directory:
   backend/data/sri_lanka_legal_corpus/

3. Required files structure:
   backend/data/sri_lanka_legal_corpus/
   ├── processed_fundamental_rights.json
   ├── fundamental_rights_articles.json
   ├── fundamental_rights_patterns.json
   └── legal_glossary_si_en_ta.json (already exists)

4. These files are specifically for FUNDAMENTAL RIGHTS, not the full constitution.

5. Update your fundamental_rights_detector.py to handle fundamental rights:
   - It will now use patterns from fundamental_rights_patterns.json
   - Semantic search will use passages from processed_fundamental_rights.json
   - Article information will come from fundamental_rights_articles.json

6. The detector will now recognize fundamental rights like:
   - Freedom of thought, conscience and religion
   - Freedom from torture
   - Equality and non-discrimination
   - Due process rights
   - Freedom of expression and association
   - Right to information
""")

# %% [code]
# Download the zip file
files.download('sri_lanka_fundamental_rights_data.zip')

Please upload your FUNDAMENTAL RIGHTS.pdf file


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saving FUNDAMENTAL RIGHTS.pdf to FUNDAMENTAL RIGHTS (1).pdf
✓ Uploaded: FUNDAMENTAL RIGHTS (1).pdf
PyMuPDF failed: document closed
pdfplumber failed: PDF.open() got an unexpected keyword argument 'strict'
✓ Extracted text using PyPDF2

✓ Extracted 23147 characters

Preview of extracted text (first 1000 characters):
--------------------------------------------------------------------------------

===== Page 1 =====
 
THE CONSTITUTION OF THE DEMOCRATIC SOCIALIST 
REPUBLIC OF SRI LANKA 
Revised Edition – 2023 
 
CHAPTERIII 
FUNDAMENTAL RIGHTS 
 
Freedom of 
thought, 
conscience 
and religion 
 
Freedom 
from torture 
 
Right to 
equality 
1. Every person is entitled to freedom of thought, 
conscience and religion, including the freedom to have or 
to adopt a religion or belief of his choice. 
2. No person shall be subjected to torture or to 
cruel, inhuman or degrading treatment or punishment. 
3. (1) All persons are equal before the law and are 
entitled to the equal protection of the la

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>